In [ ]:
import os
import sys

import importlib
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.TopoAna import rdridgelist as RL



from rnodes import rnodes, classify, synthetic_profiles
from rnodes_v2 import rnodes_v2 
from rnodes_v3 import rnodes_v3,plot_wedge

importlib.reload( RL )

In [ ]:
P = synthetic_profiles()                    # dict of three 21-point arrays
ridge = P['bump left of big mountain']  #['ridge with higher plateau on right']  #['bump between two big mountains']
ridge = P['ridge with higher plateau on right']  #['bump between two big mountains']
#ridge = P['bump between two big mountains']
#plt.plot(ridge)                             # to look at one

In [ ]:
#ridge[5]=-1.
xridge=np.arange( len(ridge) )
print( len(ridge))
print( len(xridge))
o=rnodes(ridge)
#o2=rnodes_v2( o['xnodes']-1,o['hnodes'] )
o2=rnodes_v2( xridge, ridge )


In [ ]:
plt.plot(ridge, '-+') 
plt.plot( o['xnodes']-1,o['hnodes'], '-x' )
plt.plot( o['xwedge']-1,o['hwedge'] , 'o')
plt.plot( o2['xwedge']-0,o2['hwedge']+80 , '-^')


In [ ]:

TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/

# Note only cases with v202 and higher, e.g., v203 (or created after 9/7/2026)
# have good clngt in *tiles.nc file.

#Case = 'ne30pg3_Sco100_Rw050_v203_xtra_e02'
Case = 'ne30pg3_Sco100_Rw050_v205_new_wedge_e03'
ogrid= 'ne30pg3'


# Do a file search in output directory
# Pick last (most recent) file
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
ListFile = foo[-1]
List=RL.rdridgelist( ListFile )


In [ ]:
print(List.keys())

In [ ]:
print( List.hnodes_list.shape )
print( List.rdg_profiles.shape )
print( List.hnodes_x.shape )


In [ ]:


xw = List.xwedge_list
left  = (xw[:,0] == xw[:,1])
right = (xw[:,1] == xw[:,2])
oo = np.where( (left | right) & (xw[:,1] > 0) )[0]     # any collapse, excluding the all-zero "no peak" cases
ibad=10000


In [ ]:
print( len(oo))

In [ ]:
idx_ridge = 230_401 #260_233  #oo[ibad] 
#idx_ridge = 260_233  #oo[ibad] 
#idx_ridge = 265_570  #oo[ibad] 

print( xw[idx_ridge,:] )
nsw_d = List.NSWx_diag[idx_ridge]
listridge=List.rdg_profiles[idx_ridge,:nsw_d+1 ]
xlistridge=np.arange( len(listridge) )

#listridge[28]=-1
q=rnodes(listridge)
q2=rnodes_v2( xlistridge, listridge , bound_at_higher=True )
q3=rnodes_v2( q['xnodes'] ,q['hnodes'] )

w=rnodes_v3( listridge, max_nodes_per_side=2 )

print( listridge )

print( q['xwedge'] )

d2r=180./np.pi
print( d2r*List.lonc[idx_ridge] , d2r*List.latc[idx_ridge] )
print( List.xspk[idx_ridge] , List.yspk[idx_ridge], List.MyPanel[idx_ridge] )
print( List.angll[idx_ridge] , List.anglx[idx_ridge] )

xspk,yspk,ipan = List.xspk[idx_ridge] , List.yspk[idx_ridge], List.MyPanel[idx_ridge] 
xs01,ys01 = List.xs01[idx_ridge] , List.ys01[idx_ridge]
xs,ys = List.xs[idx_ridge] , List.ys[idx_ridge]



In [ ]:
print( q['xnodes'] , q['nnode'] )

In [ ]:
fig,axs=plt.subplots(  1, 3, figsize=(3*11,7) )

nn = q['nnode'] 
ax=axs[0]
ax.plot(  List.rdg_profiles[idx_ridge,:nsw_d+1 ]  , label='Ridge profile' )
ax.plot(  List.xwedge_list[idx_ridge,:nsw_d+1 ]-1 ,List.hwedge_list[idx_ridge,: ] , 'o' , color='red' ,ms=20, label='New 3pt rms wedge' )
ax.plot( q['xnodes'][:nn]-1,q['hnodes'][:nn]+000, '--+', ms=20 , label='Original "nodes"'  )
ax.plot( q['xwedge']-1,q['hwedge']+000 , 'o',color='black',ms=10, label='Original node-based 3pt wedge' )
ax.legend()

ax=axs[1]
ax.plot(  List.rdg_profiles[idx_ridge,:nsw_d+1 ] , label='Ridge profile' )
ax.plot(  List.xwedge_list[idx_ridge,:nsw_d+1 ]-1 ,List.hwedge_list[idx_ridge,: ] , 'o' , color='red' ,ms=20, label='New 3pt rms wedge' )
ax.plot( q2['xwedge']-0,q2['hwedge']+00 , '-v', ms=15, color='gold' , label='full 5pts for rms wedge' )
#ax.plot( w['xwedge']-1, w['hwedge']+00 , '--x', ms=10, color='cyan' )
#plt.plot( q3['xwedge']-1,q3['hwedge']-70 , '-^', color='gold' )
ax.legend()

ax=axs[2]
ax.plot(  List.rdg_profiles[idx_ridge,:nsw_d+1 ] , label='Ridge profile' )
#ax.plot(  List.xwedge_list[idx_ridge,:nsw_d+1 ]-1 ,List.hwedge_list[idx_ridge,: ] , 'o' , color='red' ,ms=20, label='New 3pt rms wedge' )
ax.plot( q2['xwedge']-0,q2['hwedge']+00 , '-v', ms=15, color='gold' , label='full 5pts for rms wedge' )
ax.plot( w['xwedge']-1, w['hwedge']+00 , '--x', ms=10, color='cyan', label='full set from new "staircase"'  )
#plt.plot( q3['xwedge']-1,q3['hwedge']-70 , '-^', color='gold' )
ax.legend()

#plt.ylim( -1000,2000)

In [ ]:



fig, ax = plt.subplots(); plot_wedge(ax, listridge, w, title=f'ipk {idx_ridge}')

In [ ]:
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo*.nc')
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )


In [ ]:
x0,x1=int(xspk)-30,int(xspk)+30
y0,y1=int(yspk)-30,int(yspk)+30
print( x0,x1,y0,y1 )
print(dev[ipan,y0:y1+1,x0:x1+1].shape)
topolevs = 31 #np.linspace( listridge.min() , listridge.max() , num=21 )
fig,axs=plt.subplots(1,2,figsize=(2*11,10) )

ax=axs[0]
#ax.contourf(dev[ipan-1,y0:y1+1,x0:x1+1] , cmap='terrain', levels=topolevs)
ax.contourf(dev[ipan-1,:,:] , cmap='terrain', levels=31)
#ax.contourf(dev[ipan-1,:,:] , cmap='terrain', levels=31)
ax.scatter( xspk,yspk,s=200,marker='+')

ax=axs[1]
#ax.contourf(dev[ipan-1,y0:y1+1,x0:x1+1] , cmap='terrain', levels=topolevs)
ax.contourf(dev[ipan-1,:,:] , cmap='terrain', levels=31)
ax.set_xlim( x0, x1 )
ax.set_ylim( y0,y1 )
#ax.contourf(dev[ipan-1,:,:] , cmap='terrain', levels=31)
ax.plot( [xs,xs01,xspk],[ys,ys01,yspk], '-+', ms=20,color='red' ) 

In [ ]:
print( List.xwedge_list[idx_ridge,: ], q['xwedge'] )

In [ ]:
print( List.NSWx_diag[idx_ridge] )
P=List.rdg_profiles
nsw=List.NSWx_diag

In [ ]:
print(P.shape,nsw.shape)
npks,nsw_fix=P.shape
ik = 200_000
plt.plot( P[ik,0:nsw[ik]+1 ] )

In [ ]:
r3 = []; r5 = []
for ik in np.arange( npks ):
    #for p in P[:, :nsw+1]:
    p=P[ik,0:nsw[ik]+1]
    nswx=nsw[ik]
    w = rnodes_v2(np.arange(1,nswx+2), p, nnode=nswx+1, ns2=nswx//2, bound_at_higher=True)
    if w['ok']:
        r3.append(w['hwedge3'].max() - w['hwedge3'].min())
        r5.append(w['hwedge5'].max() - w['hwedge5'].min())
    if ik%10000==0:
        print( ik )
r3, r5 = np.array(r3), np.array(r5)
ratio = r5 / r3
print(np.percentile(ratio, [50, 75, 90, 99]), (ratio > 1.2).mean())

In [ ]:
print(List.keys())